# Demo: `backtest()` with LLM-powered explanations

This notebook demonstrates the `ForecastingAssistant.backtest()` method
combined with the `ask()` method to get natural-language explanations
of backtesting results using a Google Gemini LLM.

The workflow:
1. Configure the assistant with a Google API key
2. Profile the data
3. Generate a forecasting plan
4. Create a cross-validation strategy
5. Run backtesting
6. Use `ask()` to get LLM-powered explanations of the results

**Requirements:** A valid Google API key with access to Gemini models.

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
import warnings

import numpy as np
import pandas as pd
from skforecast.model_selection import TimeSeriesFold

from skforecast_ai import ForecastingAssistant

warnings.filterwarnings("ignore")

## 1. Configure the Assistant with Google Gemini

Replace the placeholder below with your actual Google API key.

In [ ]:
GOOGLE_API_KEY = "your-google-api-key-here"  # Replace with your actual Google API key

In [ ]:
assistant = ForecastingAssistant(
    llm="google:gemini-3-flash-preview",
    api_key=GOOGLE_API_KEY,
)

## 2. Synthetic Data

Create sample datasets for single-series and multi-series backtesting.

In [ ]:
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")

# Single series target with trend + seasonality
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)

# Exogenous variables
promo = rng.normal(50, 10, n)
temperature = 15 + 10 * np.sin(2 * np.pi * np.arange(n) / 365) + rng.normal(0, 2, n)

# Single series with exogenous variables
df = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": promo,
    "temperature": temperature,
})

# Multi-series (long format)
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "series_id": ["store_A"] * n_multi + ["store_B"] * n_multi + ["store_C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
    "promo": rng.normal(50, 10, n_multi * 3),
})

print(f"df:       {df.shape}")
print(f"df_multi: {df_multi.shape}")
display(df.head())

---
## 3. Single Series Backtesting + LLM Explanation

Run backtesting on a single series with `ForecasterRecursive`, then use `ask()` to get an LLM-generated interpretation of the results.

### 3.1 Profile, Plan, and Backtest

In [ ]:
profile = assistant.profile(
    data        = df,
    target      = "sales",
    date_column = "date",
)

plan = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
)

cv, cv_explanation = assistant.create_cv(
    profile = profile,
    plan    = plan,
)

print("CV explanation:")
print(cv_explanation)
print(f"\nCV: {cv}")

In [ ]:
result = assistant.backtest(
    data        = df,
    target      = "sales",
    date_column = "date",
    cv          = cv,
    profile     = profile,
    plan        = plan,
)

print("Explanation:")
print(result.explanation)
print("\nMetrics:")
display(result.metrics)
print("\nPredictions (first rows):")
display(result.predictions.head(10))

In [ ]:
print(result.code)

### 3.2 Ask the LLM about the backtesting results

Pass the `backtest_result` to `ask()` — the LLM receives metrics, predictions, and CV configuration in context.

In [ ]:
answer = assistant.ask(
    prompt="Analyze the backtesting results. Are the metrics good? "
           "What does the error distribution suggest about the model?",
    backtest_result=result,
)
print(answer.response)

In [ ]:
answer = assistant.ask(
    prompt="Based on these backtesting results, what would you recommend "
           "to improve the forecast accuracy? Should I try different lags, "
           "add more features, or switch to a different model?",
    backtest_result=result,
)
print(answer.response)

---
## 4. Multi-Series Backtesting + LLM Explanation

Run backtesting on multiple series with `ForecasterRecursiveMultiSeries`.

In [ ]:
profile_multi = assistant.profile(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
)

plan_multi = assistant.plan(
    profile    = profile_multi,
    steps      = 7,
    forecaster = "ForecasterRecursiveMultiSeries",
    estimator  = "LGBMRegressor",
)

cv_multi, _ = assistant.create_cv(
    profile = profile_multi,
    plan    = plan_multi,
)

result_multi = assistant.backtest(
    data             = df_multi,
    target           = "revenue",
    date_column      = "date",
    series_id_column = "series_id",
    cv               = cv_multi,
    profile          = profile_multi,
    plan             = plan_multi,
)

print("Metrics (per series):")
display(result_multi.metrics)
print("\nPredictions (first rows):")
display(result_multi.predictions.head(10))

In [ ]:
answer_multi = assistant.ask(
    prompt="Compare the backtesting performance across the three stores. "
           "Which store is hardest to forecast and why might that be?",
    backtest_result=result_multi,
)
print(answer_multi.response)

---
## 5. General Forecasting Q&A (no data)

The `ask()` method can also answer general skforecast questions without any data context.

In [ ]:
answer_general = assistant.ask(
    prompt="What is the difference between ForecasterRecursive and "
           "ForecasterDirect? When should I use each one?",
)
print(answer_general.response)

In [ ]:
answer_intervals = assistant.ask(
    prompt="How do I add prediction intervals to my backtesting? "
           "What methods are available in skforecast?",
)
print(answer_intervals.response)

---
## 6. Backtesting with Prediction Intervals + LLM Analysis

Run backtesting with prediction intervals and ask the LLM to evaluate coverage.

In [ ]:
plan_intervals = assistant.plan(
    profile    = profile,
    steps      = 14,
    forecaster = "ForecasterRecursive",
    estimator  = "LGBMRegressor",
    interval   = [10, 90],
)

cv_intervals, _ = assistant.create_cv(
    profile = profile,
    plan    = plan_intervals,
)

result_intervals = assistant.backtest(
    data        = df,
    target      = "sales",
    date_column = "date",
    cv          = cv_intervals,
    profile     = profile,
    plan        = plan_intervals,
)

print("Metrics:")
display(result_intervals.metrics)
print("\nPredictions with intervals (first rows):")
display(result_intervals.predictions.head(10))

In [ ]:
answer_coverage = assistant.ask(
    prompt="Evaluate the prediction intervals. Is the coverage close "
           "to the nominal 80%? Are the intervals too wide or too narrow?",
    backtest_result=result_intervals,
)
print(answer_coverage.response)

---
## 7. Shortcut: Fully Automatic Backtesting

The `backtest()` method can handle everything automatically — profiling, plan generation, and execution — with just the data and a `TimeSeriesFold`.

In [ ]:
cv_simple = TimeSeriesFold(
    steps=14,
    initial_train_size=300,
    refit=False,
    fixed_train_size=False,
)

result_auto = assistant.backtest(
    data        = df,
    target      = "sales",
    date_column = "date",
    cv          = cv_simple,
)

print(f"Auto-selected forecaster: {result_auto.plan.forecaster}")
print(f"Auto-selected estimator: {result_auto.plan.estimator}")
print(f"Uses exog: {result_auto.plan.use_exog}")
print("\nMetrics:")
display(result_auto.metrics)

In [ ]:
answer_auto = assistant.ask(
    prompt="Summarize the full backtesting pipeline: what forecaster was chosen, "
           "what features were used, and how did it perform?",
    backtest_result=result_auto,
)
print(answer_auto.response)